# 🛰️ GeoNexa — RemoteCLIP Foundation Model Fine-Tuning
### Contrastive Learning on Real 27,000 Sentinel-2 Optical Satellite Imagery (SIH 2026)

This notebook fine-tunes **RemoteCLIP (ViT-B-32)** on real optical satellite imagery (EuroSAT / Sentinel-2) across 10 land-cover categories on a GPU (T4 / A100 / V100).

---
### Workflow:
1. **Environment Setup**: Clone repository & install dependencies
2. **Download Weights & Real Sentinel-2 Data**: Download base RemoteCLIP weights (`RemoteCLIP-ViT-B-32.pt`) and EuroSAT optical satellite dataset
3. **Generate Image-Text Contrastive Pairs**: 25,200 training/validation pairs
4. **GPU Contrastive Fine-Tuning**: 15-30 Epochs with InfoNCE loss + Cosine LR Schedule + AdamW on GPU
5. **WiSE-FT Weight Interpolation & Export**: Export deployment-ready model and auto-download to your local PC

## 1. Environment & GPU Verification

In [ ]:
!nvidia-smi
# Remove old clone if re-running
!rm -rf GeoNexa
!git clone https://github.com/Amityush-lgtm/GeoNexa.git
%cd GeoNexa
!pip install -q open-clip-torch rasterio scipy pillow torchvision tqdm gdown faiss-cpu pydantic pydantic-settings fastapi uvicorn

## 2. Download Pretrained RemoteCLIP Foundation Weights

In [ ]:
!python scripts/download_models.py --force

## 3. Download Real Sentinel-2 Satellite Dataset & Generate Pairs

In [ ]:
import torchvision.datasets
print('Downloading real Sentinel-2 EuroSAT dataset (27,000 satellite images)...')
ds = torchvision.datasets.EuroSAT(root='data/eurosat', download=True)
print(f'Successfully downloaded {len(ds)} real Sentinel-2 optical images across classes:', ds.classes)

# Generate 25,200 domain-specific contrastive training & validation pairs
!python scripts/prepare_eurosat_pairs.py

## 4. Run GPU Contrastive Fine-Tuning

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on device: {device}')

# Run 15-epoch fine-tuning on GPU with batch size 32
!python -m training.train \
    --train-data training_data/train_pairs.json \
    --val-data training_data/val_pairs.json \
    --checkpoint models/remoteclip-vit-b-32/RemoteCLIP-ViT-B-32.pt \
    --output-dir checkpoints \
    --epochs 15 \
    --batch-size 32 \
    --lr 1e-5 \
    --device cuda \
    --unfreeze-last-n 4 \
    --num-workers 2 \
    --experiment-name geonexa_gpu_production

## 5. Export Fine-Tuned Model with WiSE-FT Weight Interpolation & Download

In [ ]:
# Export with WiSE-FT (Weight-Space Ensemble) blending fine-tuned + base zero-shot representations
ckpt_file = 'checkpoints/final_model.pt'
!python scripts/export_finetuned_model.py \
    --checkpoint {ckpt_file} \
    --base-model models/remoteclip-vit-b-32/RemoteCLIP-ViT-B-32.pt \
    --alpha 0.4 \
    --output models/RemoteCLIP-ViT-B-32-finetuned.pt \
    --verify

# Download the exported model weights directly to your local PC
from google.colab import files
files.download('models/RemoteCLIP-ViT-B-32-finetuned.pt')
files.download('checkpoints/training_report.json')

## 6. How to Deploy to Your Local GeoNexa Backend

Once downloaded to your PC:
1. Move `RemoteCLIP-ViT-B-32-finetuned.pt` into `d:\SIH 2.0\semantic-eo-search\models\`
2. Rebuild the FAISS index:
   ```bash
   python scripts/build_index.py --model-path models/RemoteCLIP-ViT-B-32-finetuned.pt --force-rebuild
   ```
3. Run retrieval evaluation:
   ```bash
   python scripts/evaluate_retrieval.py
   ```